In [ ]:
from ultralytics import YOLO
import cv2

print("🚀 正在加载 INT8 量化边缘模型...")
model = YOLO("../../runs/detect/train-2/weights/best_int8.onnx", task='detect')

print("👉 正在调取摄像头，按 'q' 键退出...")
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success: 
        break
    
    results = model.predict(source=frame, stream=True, verbose=False, conf=0.15)
    for r in results:
        annotated_frame = r.plot()
        class_names = [r.names[int(cls)] for cls in r.boxes.cls]
        
        status = "Mask Detected (SAFE)" if 'face_mask' in class_names else "No Mask (DANGER)"
        color = (0, 255, 0) if status == "Mask Detected (SAFE)" else (0, 0, 255)
        
        cv2.putText(annotated_frame, status, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.imshow("Interactive Demo", annotated_frame)
        
    if cv2.waitKey(1) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()
